# Livrable 2 :

In [1]:
import tensorflow as tf
import numpy as np
import os
import sys
from tensorflow import keras

project_root = os.path.abspath(os.path.join(".."))  # One level up from current script
if project_root not in sys.path:
    sys.path.append(project_root)

from src.model_loader import ModelLoader
from src.data_loader import DataLoader

print(tf.__version__)

2.16.2


## Généation du dataset de photos

In [2]:
data_loader = DataLoader()
train_ds, test_ds, val_ds = data_loader.load_unique_dataset(folder_name="Photo")


Photo Dataset size: 9993


2025-04-18 15:07:51.064144: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-04-18 15:07:51.064176: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-04-18 15:07:51.064181: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-04-18 15:07:51.064201: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-18 15:07:51.064212: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## Bruitage du dataset

In [3]:
noisy_train_list = list(data_loader.add_noise_to_dataset(
    dataset=train_ds,
    noise_factor_range=(0.1, 0.6),
    pixel_noise_prob=0.4))

noisy_val_list = list(data_loader.add_noise_to_dataset(
    dataset=val_ds,
    noise_factor_range=(0.1, 0.6),
    pixel_noise_prob=0.4))

noisy_test_list = list(data_loader.add_noise_to_dataset(
    dataset=test_ds,
    noise_factor_range=(0.1, 0.6),
    pixel_noise_prob=0.4))

2025-04-18 15:08:05.290188: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-04-18 15:08:06.766671: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-04-18 15:08:07.992113: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
# 1. Générer les images bruitées une fois
noisy_train_list = list(noisy_train_list)
noisy_val_list = list(noisy_val_list)
noisy_test_list = list(noisy_test_list)

In [5]:
# Aplatir la liste de batches en liste d’images individuelles
flattened_noisy_list_train = []
for batch in noisy_train_list:
    for x, y in zip(*batch):
        flattened_noisy_list_train.append((x, y))

flattened_noisy_list_val = []
for batch in noisy_val_list:
    for x, y in zip(*batch):
        flattened_noisy_list_val.append((x, y))

flattened_noisy_list_test = []
for batch in noisy_test_list:
    for x, y in zip(*batch):
        flattened_noisy_list_val.append((x, y))

In [6]:
noisy_train_ds = tf.data.Dataset.from_generator(
    lambda: iter(flattened_noisy_list_train),
    output_signature=(
        tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)  # ou tf.string selon ton label
    )
).batch(32, drop_remainder=True).cache()

noisy_val_ds = tf.data.Dataset.from_generator(
    lambda: iter(flattened_noisy_list_val),
    output_signature=(
        tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)  # ou tf.string selon ton label
    )
).batch(32, drop_remainder=True).cache()

noisy_test_ds = tf.data.Dataset.from_generator(
    lambda: iter(flattened_noisy_list_test),
    output_signature=(
        tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)  # ou tf.string selon ton label
    )
).batch(32, drop_remainder=True).cache()

In [7]:
noisy_train_ds.element_spec
train_ds.element_spec

(TensorSpec(shape=(32, 256, 256, 3), dtype=tf.float32, name=None),
 TensorSpec(shape=(32,), dtype=tf.int64, name=None))

In [ ]:
from src.utils import show_clean_and_noisy_images

show_clean_and_noisy_images(train_ds, noisy_train_ds, max_images=10, batch_index=10)

In [ ]:
from src.utils import show_clean_and_noisy_images

show_clean_and_noisy_images(train_ds, noisy_train_ds, max_images=10, batch_index=10)

In [8]:
@tf.autograph.experimental.do_not_convert
def make_autoencoder_dataset(input_ds, target_ds):
    return tf.data.Dataset.zip((input_ds.map(lambda x, y: x), target_ds.map(lambda x, y: x)))

In [9]:
train_autoenc_ds = make_autoencoder_dataset(noisy_train_ds, train_ds)
val_autoenc_ds = make_autoencoder_dataset(noisy_val_ds, val_ds)

In [13]:
autoencoder_loader_base = ModelLoader(model_name="autoencoder_base_mae")
autoencoder_base = autoencoder_loader_base.create_base_autoencoder(show_summary=True)

autoencoder_loader_sl = ModelLoader(model_name="skip_layers_mae")
autoencoder_sl = autoencoder_loader_sl.create_skip_layer_autoencoder(show_summary=True)

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 8, 8, 256)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 981,440 (3.74 MB)

 Trainable params: 979,968 (3.74 MB)

 Non-trainable params: 1,472 (5.75 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder             │ [(None, 8, 8,     │    981,440 │ input_layer_2[0]… │
│ (Functional)        │ 256), (None, 16,  │            │                   │
│                     │ 16, 256), (None,  │            │                   │
│                     │ 32, 32, 128),     │            │                   │
│                     │ (None, 64, 64,    │            │                   │
│                     │ 64), (None, 128,  │            │                   │
│                     │ 128, 32)]         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder             │ (None, 256, 256,  │  1,387,619 │ encoder[0][0],    │
│ (Functional)        │ 3)                │            │ encoder[0][1],    │
│                     │                   │            │ encoder[0][2],    │
│                     │                   │            │ encoder[0][3],    │
│                     │                   │            │ encoder[0][4]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,369,059 (9.04 MB)

 Trainable params: 2,365,667 (9.02 MB)

 Non-trainable params: 3,392 (13.25 KB)

Autoencoder created successfully.


In [14]:
# Train the model
with tf.device("/gpu:0"):
    history = autoencoder_base.fit(train_autoenc_ds,
                    epochs=1,
                    batch_size= 32,
                    validation_data=val_autoenc_ds,
                    verbose=2
                   )

2025-04-18 15:09:22.044383: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-04-18 15:10:38.011173: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-04-18 15:10:38.012731: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/adam/Add_14/_100]]
2025-04-18 15:10:38.012745: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14802265465025200665
2025-04-18 15:10:38.012750: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9205182624475136648
2025-04-18 15:10:38.012757: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 86

199/199 - 90s - 450ms/step - loss: 0.0914 - mae: 0.0914 - psnr_metric: 18.2128 - ssim_metric: 0.3988 - val_loss: 0.1341 - val_mae: 0.1341 - val_psnr_metric: 15.5364 - val_ssim_metric: 0.3779


2025-04-18 15:10:50.762891: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-04-18 15:10:50.836175: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-04-18 15:10:50.836199: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2025-04-18 15:10:50.836206: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8832852015714811043
2025-04-18 15:10:50.836209: I tens

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='test')
plt.legend()


In [ ]:
import matplotlib.pyplot as plt

# Tracer l'évolution de MAE
plt.figure(figsize=(12, 8))
plt.plot(history.history['mae'], label='train MAE')
plt.plot(history.history['val_mae'], label='val MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error (MAE)')
plt.title('MAE over Epochs')
plt.legend()
plt.show()

# Tracer l'évolution de SSIM
plt.figure(figsize=(12, 8))
plt.plot(history.history['ssim_metric'], label='train SSIM')
plt.plot(history.history['val_ssim_metric'], label='val SSIM')
plt.xlabel('Epochs')
plt.ylabel('Structural Similarity (SSIM)')
plt.title('SSIM over Epochs')
plt.legend()
plt.show()

# Tracer l'évolution de PSNR
plt.figure(figsize=(12, 8))
plt.plot(history.history['psnr_metric'], label='train PSNR')
plt.plot(history.history['val_psnr_metric'], label='val PSNR')
plt.xlabel('Epochs')
plt.ylabel('Peak Signal-to-Noise Ratio (PSNR)')
plt.title('PSNR over Epochs')
plt.legend()
plt.show()

In [ ]:
import cv2

def variance_of_laplacian_rgb(image_rgb):
    image_rgb = np.array(image_rgb)  # Assure que c'est un array numpy
    if image_rgb.shape[-1] == 3:
        gray = cv2.cvtColor((image_rgb * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image_rgb * 255).astype(np.uint8)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    return laplacian.var()

def tenengrad_rgb(image_rgb):
    image_rgb = np.array(image_rgb)  # Assure que c'est un array numpy
    if image_rgb.shape[-1] == 3:
        gray = cv2.cvtColor((image_rgb * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image_rgb * 255).astype(np.uint8)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1)
    magnitude = np.sqrt(gx**2 + gy**2)
    return np.mean(magnitude)

In [ ]:
import matplotlib.pyplot as plt


# Fonction SSIM
def ssim_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

# Fonction PSNR
def psnr_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

# Fonction MAE
def mae_metric(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))

def show_full_prediction_metrics(noisy_dataset, clean_dataset, model, n=10):
    import matplotlib.pyplot as plt

    # Toutes les métriques globales
    lap_sharpness_list = []
    ten_sharpness_list = []
    mae_list = []
    ssim_list = []
    psnr_list = []

    all_noisy = []
    all_clean = []
    all_decoded = []

    print("📊 Prédiction sur l'ensemble du dataset...")
    for (noisy_batch, _), (clean_batch, _) in zip(noisy_dataset, clean_dataset):
        decoded_batch = model.predict(noisy_batch, verbose=0)
        for i in range(len(noisy_batch)):
            noisy = noisy_batch[i].numpy()
            original = clean_batch[i].numpy()
            decoded = decoded_batch[i]

            # Métriques
            sharp_orig_lap = variance_of_laplacian_rgb(original)
            sharp_dec_lap = variance_of_laplacian_rgb(decoded)

            sharp_orig_ten = tenengrad_rgb(original)
            sharp_dec_ten = tenengrad_rgb(decoded)

            ssim_val = ssim_metric(original, decoded).numpy()
            psnr_val = psnr_metric(original, decoded).numpy()
            mae_val = mae_metric(original, decoded).numpy()

            lap_percent = (sharp_dec_lap / sharp_orig_lap) * 100 if sharp_orig_lap != 0 else 0
            ten_percent = (sharp_dec_ten / sharp_orig_ten) * 100 if sharp_orig_ten != 0 else 0

            lap_sharpness_list.append(lap_percent)
            ten_sharpness_list.append(ten_percent)
            mae_list.append(mae_val)
            ssim_list.append(ssim_val)
            psnr_list.append(psnr_val)
            
            # Stocker pour affichage des exemples
            if len(all_decoded) < n:
                all_noisy.append(noisy)
                all_clean.append(original)
                all_decoded.append(decoded)

    # --- Affichage de n exemples ---
    plt.figure(figsize=(18, 12))
    for i in range(n):
        noisy = all_noisy[i]
        original = all_clean[i]
        decoded = all_decoded[i]

        sharp_orig_lap = variance_of_laplacian_rgb(original)
        sharp_dec_lap = variance_of_laplacian_rgb(decoded)

        sharp_orig_ten = tenengrad_rgb(original)
        sharp_dec_ten = tenengrad_rgb(decoded)

        ssim_val = ssim_metric(original, decoded).numpy()
        psnr_val = psnr_metric(original, decoded).numpy()
        mae_val = mae_metric(original, decoded).numpy()

        sharpness_percent_lap = (sharp_dec_lap / sharp_orig_lap) * 100 if sharp_orig_lap != 0 else 0
        sharpness_percent_ten = (sharp_dec_ten / sharp_orig_ten) * 100 if sharp_orig_ten != 0 else 0

        # Image bruitée
        ax = plt.subplot(5, n, i + 1)
        plt.imshow(noisy)
        plt.title("Bruitée")
        plt.axis("off")

        # Originale
        ax = plt.subplot(5, n, n + i + 1)
        plt.imshow(original)
        plt.title(f"Originale\nLap: {sharp_orig_lap:.1f} | Ten: {sharp_orig_ten:.1f}")
        plt.axis("off")

        # Reconstruite
        ax = plt.subplot(5, n, 2 * n + i + 1)
        plt.imshow(decoded)
        plt.title(
            f"Reconstruit\nLap: {sharp_dec_lap:.1f} ({sharpness_percent_lap:.1f}%)\n"
            f"Ten: {sharp_dec_ten:.1f} ({sharpness_percent_ten:.1f}%)\n"
            f"SSIM: {ssim_val:.3f} | PSNR: {psnr_val:.1f}\nMAE: {mae_val:.4f}"
        )
        plt.axis("off")

    # --- Moyennes globales ---
    avg_lap = sum(lap_sharpness_list) / len(lap_sharpness_list)
    avg_ten = sum(ten_sharpness_list) / len(ten_sharpness_list)
    avg_mae = sum(mae_list) / len(mae_list)
    avg_ssim = sum(ssim_list) / len(ssim_list)
    avg_psnr = sum(psnr_list) / len(psnr_list)

    ax = plt.subplot(5, 1, 5)
    plt.axis('off')
    plt.text(
        0.5, 0.5,
        (
            f"**Moyennes globales sur {len(lap_sharpness_list)} images**\n\n"
            f"Netteté Laplacian: {avg_lap:.2f}% | Tenengrad: {avg_ten:.2f}%\n"
            f"MAE: {avg_mae:.4f} | SSIM: {avg_ssim:.3f} | PSNR: {avg_psnr:.1f} dB"
        ),
        fontsize=16, ha='center', va='center', fontweight='bold'
    )

    plt.tight_layout()
    plt.show()


In [ ]:

import datetime

# Récupérer la date et l'heure actuelle
now = datetime.datetime.now()
timestamp = now.strftime("%d%m%Y_%H%M")  # Format : jour mois année _ heure minute

# Construire le chemin de sauvegarde
save_path = f"./../models/autoencoder_skip_layer_mae_{timestamp}.weights.h5"

# Sauvegarder
autoencoder.save_weights(save_path)

In [ ]:
show_full_prediction_metrics(noisy_test_ds, test_ds, autoencoder)

In [ ]:
show_clean_and_noisy_images(test_ds, noisy_test_ds, max_images=10)